# Apache Beam Lab 1: Environment Setup

## Overview
This lab will guide you through setting up your Apache Beam development environment. A properly configured environment is essential for developing and testing data pipelines effectively.

## Learning Objectives
- Install Apache Beam and required dependencies
- Set up Python virtual environment
- Configure pipeline runners (DirectRunner, DataflowRunner)
- Validate installation with test pipeline
- Understand runner options and trade-offs

## Prerequisites
- Python 3.8 or higher
- pip (Python package manager)
- Basic command line familiarity

## Step 1: System Requirements Check
Let's verify your system meets the requirements.

In [ ]:
# Check Python version
import sys
print(f"Python version: {sys.version}")
print(f"Python version required: 3.8+")
print(f"Requirement met: {sys.version_info >= (3, 8)}")

## Step 2: Install Dependencies

Install Apache Beam and its dependencies using the requirements file.

In [ ]:
# Install Apache Beam with core dependencies
import subprocess
import os

os.chdir('/home/ramdov/projects/beam-code-practice')
result = subprocess.run(['pip', 'install', '-r', 'requirements.txt'], 
                      capture_output=True, 
                      text=True)
print(result.stdout)
if result.returncode != 0:
    print("Installation errors:")
    print(result.stderr)
else:
    print("Installation successful!")

## Step 3: Verify Installation

Let's verify that Apache Beam is properly installed.

In [ ]:
# Import Apache Beam and check version
import apache_beam as beam
print(f"Apache Beam version: {beam.__version__}")

# Check available runners
from apache_beam.runners.runner import PipelineRunner
print(f"Pipeline runners available: {beam.PipelineRunner.__subclasses__()}")

## Step 4: Create Your First Pipeline

Let's create a simple test pipeline to verify everything is working correctly.

In [ ]:
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions

# Define a simple pipeline
def run_test_pipeline():
    """Run a simple test pipeline."""
    with beam.Pipeline() as pipeline:
        (pipeline 
         | 'Create data' >> beam.Create([1, 2, 3, 4, 5])
         | 'Multiply by 2' >> beam.Map(lambda x: x * 2)
         | 'Print results' >> beam.Map(print))

print("Running test pipeline...")
run_test_pipeline()
print("Test pipeline completed successfully!")

## Step 5: Pipeline Options Configuration

Understanding pipeline options is crucial for configuring how your pipeline runs.

In [ ]:
from apache_beam.options.pipeline_options import PipelineOptions, StandardOptions

# Create pipeline options
options = PipelineOptions()

# Set runner (DirectRunner for local testing)
options.view_as(StandardOptions).runner = 'DirectRunner'

# Display current options
print("Pipeline Options:")
for attr in dir(options):
    if not attr.startswith('_'):
        try:
            value = getattr(options, attr)
            if not callable(value):
                print(f"  {attr}: {value}")
        except:
            pass

## Step 6: Understanding Pipeline Runners

Apache Beam supports multiple runners, each with different use cases:

In [ ]:
# Display information about different runners
runners_info = {
    'DirectRunner': {
        'description': 'Local execution for testing and development',
        'use_case': 'Development, testing, small datasets',
        'scalability': 'Limited to single machine',
        'setup': 'No additional setup required'
    },
    'DataflowRunner': {
        'description': 'Google Cloud managed service',
        'use_case': 'Production workloads on Google Cloud',
        'scalability': 'Auto-scaling, managed infrastructure',
        'setup': 'Requires Google Cloud project and credentials'
    },
    'SparkRunner': {
        'description': 'Apache Spark execution engine',
        'use_case': 'On-premises or cloud Spark clusters',
        'scalability': 'Depends on Spark cluster size',
        'setup': 'Requires Spark cluster setup'
    },
    'FlinkRunner': {
        'description': 'Apache Flink execution engine',
        'use_case': 'Streaming-focused workloads',
        'scalability': 'Depends on Flink cluster size',
        'setup': 'Requires Flink cluster setup'
    }
}

print("Apache Beam Pipeline Runners:")
for runner, info in runners_info.items():
    print(f"\n{runner}:")
    for key, value in info.items():
        print(f"  {key}: {value}")

## Step 7: Environment Variables Setup

Set up environment variables for pipeline configuration.

In [ ]:
import os

# Set environment variables for Beam
os.environ['BEAM_RUNNER'] = 'DirectRunner'
os.environ['BEAM_PIPELINE'] = 'test_pipeline'

# Display environment variables
print("Environment Variables:")
beam_vars = {k: v for k, v in os.environ.items() if 'BEAM' in k}
for var, value in beam_vars.items():
    print(f"  {var}: {value}")

## Step 8: Working with Sample Data

Let's test your setup by processing the sample data from Lab 0.

In [ ]:
import apache_beam as beam
import pandas as pd

# Load sample data
sales_file = '/home/ramdov/projects/beam-code-practice/data/sample_sales.csv'
if os.path.exists(sales_file):
    df = pd.read_csv(sales_file)
    print(f"Loaded {len(df)} sales records")
    print(f"\nSample data:")
    print(df.head(3))
else:
    print("Sample data file not found. Run Lab 0 first.")

## Exercise: Create a Data Processing Pipeline

### Task 1: Basic Data Processing
Create a pipeline that reads the sample sales data and calculates total sales by product.

In [ ]:
# Your code here to create a pipeline that processes sales data
# 1. Read the CSV data
# 2. Calculate total sales by product
# 3. Output the results

import apache_beam as beam
import pandas as pd

def calculate_sales_by_product():
    """Calculate total sales by product using Apache Beam."""
    # Load data
    sales_file = '/home/ramdov/projects/beam-code-practice/data/sample_sales.csv'
    df = pd.read_csv(sales_file)
    
    # Convert to list of dicts for Beam
    data = df.to_dict('records')
    
    with beam.Pipeline() as pipeline:
        results = (
            pipeline
            | 'Create data' >> beam.Create(data)
            | 'Extract product and total' >> beam.Map(
                lambda x: (x['product_id'], x['quantity'] * x['price']))
            | 'Sum by product' >> beam.CombinePerKey(sum)
            | 'Format results' >> beam.Map(
                lambda x: f"Product {x[0]}: ${x[1]:.2f}")
            | 'Print' >> beam.Map(print)
        )

if os.path.exists(sales_file):
    print("Calculating sales by product...")
    calculate_sales_by_product()
else:
    print("Sample data not found. Please run Lab 0 first.")

### Task 2: Filter and Transform Data
Create a pipeline that filters sales above a certain threshold and applies transformations.

In [ ]:
# Your code here to filter and transform sales data
# 1. Filter transactions over $100
 # 2. Apply a 10% discount
# 3. Count transactions by customer

def filter_and_transform_sales():
    """Filter and transform sales data."""
    sales_file = '/home/ramdov/projects/beam-code-practice/data/sample_sales.csv'
    df = pd.read_csv(sales_file)
    data = df.to_dict('records')
    
    with beam.Pipeline() as pipeline:
        (
            pipeline
            | 'Create data' >> beam.Create(data)
            | 'Filter high value' >> beam.Filter(
                lambda x: x['quantity'] * x['price'] > 100)
            | 'Apply discount' >> beam.Map(
                lambda x: {**x, 'discounted_total': x['quantity'] * x['price'] * 0.9})
            | 'Extract customer' >> beam.Map(
                lambda x: (x['customer_id'], 1))
            | 'Count by customer' >> beam.CombinePerKey(sum)
            | 'Format' >> beam.Map(
                lambda x: f"Customer {x[0]}: {x[1]} high-value transactions")
            | 'Print' >> beam.Map(print)
        )

if os.path.exists(sales_file):
    print("Filtering and transforming sales data...")
    filter_and_transform_sales()
else:
    print("Sample data not found. Please run Lab 0 first.")

## Summary

In this lab, you have:
- Installed Apache Beam and required dependencies
- Verified your installation with test pipelines
- Configured pipeline options and runners
- Created basic data processing pipelines
- Applied filtering and transformations to real data

Your environment is now ready for more advanced Apache Beam concepts.

## Next Steps

Proceed to [Lab 2: Pipeline Fundamentals](lab-02-pipeline-fundamentals.md) to learn about core pipeline concepts and patterns.